# ETL Pipeline - E-commerce Data (Memory Optimized)

**Pipeline:** Raw Data → Extract → Transform → Load

---

## 1. Cấu hình & Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import sqlite3
import gc
import warnings
warnings.filterwarnings('ignore')

# Config
class Config:
    DATA_DIR = Path('data')
    LOG_DIR = Path('logs')
    OUTPUT_DIR = Path('data_clean')
    DB_PATH = Path('data/bigdata.db')  # SQLite database
    
    CSV_FILES = [
        '2019-Oct.csv', '2019-Nov.csv', '2019-Dec.csv',
        '2020-Jan.csv', '2020-Feb.csv'
    ]

Config.LOG_DIR.mkdir(exist_ok=True)
Config.OUTPUT_DIR.mkdir(exist_ok=True)
Config.DB_PATH.parent.mkdir(exist_ok=True)

# SQLite setup
def get_db_connection():
    conn = sqlite3.connect(Config.DB_PATH)
    return conn

def init_database():
    conn = get_db_connection()
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS events (
            event_id INTEGER PRIMARY KEY AUTOINCREMENT,
            event_time TEXT,
            event_type TEXT,
            product_id INTEGER,
            category_id REAL,
            category_code TEXT,
            brand TEXT,
            price REAL,
            user_id INTEGER,
            user_session TEXT,
            source_file TEXT,
            date TEXT,
            hour INTEGER,
            day_of_week INTEGER,
            day_name TEXT,
            month INTEGER,
            year INTEGER,
            category_level1 TEXT,
            category_level2 TEXT,
            category_level3 TEXT
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS pipeline_metadata (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            table_name TEXT,
            row_count INTEGER,
            processed_at TEXT,
            status TEXT
        )
    ''')
    
    conn.commit()
    conn.close()
    print(f'Database initialized: {Config.DB_PATH}')

init_database()
print('Setup complete!')

Database initialized: data\bigdata.db
Setup complete!


## 2. EXTRACT + TRANSFORM + LOAD (Chunk by Chunk)

In [2]:
# Define dtypes to reduce memory
DTYPES = {
    'event_type': 'category',
    'product_id': 'int32',
    'category_id': 'float32',
    'category_code': 'string',
    'brand': 'string',
    'price': 'float32',
    'user_id': 'int32',
    'user_session': 'string'
}

# Statistics tracking
stats = {
    'total_rows': 0,
    'duplicates_removed': 0,
    'missing_handled': 0,
    'outliers_removed': 0,
    'sessions_validated': 0
}

conn = get_db_connection()

for csv_file in Config.CSV_FILES:
    file_path = Config.DATA_DIR / csv_file
    if not file_path.exists():
        print(f'Skipping {csv_file} - not found')
        continue
    
    print(f'\nProcessing: {csv_file}')
    
    # Process in chunks
    chunk_size = 100000
    total_rows_file = 0
    
    for chunk_num, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, low_memory=False, dtype=DTYPES)):
        # 1. Remove duplicates
        before = len(chunk)
        chunk = chunk.drop_duplicates()
        stats['duplicates_removed'] += before - len(chunk)
        
        # 2. Handle missing values
        chunk['category_code'] = chunk['category_code'].fillna('unknown')
        chunk['brand'] = chunk['brand'].fillna('unknown')
        chunk['user_session'] = chunk['user_session'].fillna(f'temp_s_{chunk_num}')
        chunk['price'] = chunk['price'].fillna(chunk['price'].median())
        chunk = chunk.dropna(subset=['event_time'])
        
        # 3. Standardize
        chunk['category_code'] = chunk['category_code'].str.lower().str.strip()
        chunk['brand'] = chunk['brand'].str.lower().str.strip()
        chunk.loc[chunk['category_code'] == '', 'category_code'] = 'unknown'
        chunk.loc[chunk['brand'] == '', 'brand'] = 'unknown'
        
        # 4. Validate sessions
        before_sess = len(chunk)
        chunk = chunk[chunk['user_session'].notna()]
        chunk = chunk[chunk['user_session'].str.len() >= 10]
        stats['sessions_validated'] += before_sess - len(chunk)
        
        # 5. Parse datetime efficiently
        chunk['event_time'] = pd.to_datetime(chunk['event_time'], errors='coerce')
        chunk = chunk[chunk['event_time'].notna()]
        chunk = chunk[(chunk['event_time'] >= '2019-10-01') & (chunk['event_time'] <= '2020-02-29')]
        
        # 6. Remove price outliers
        before_price = len(chunk)
        q01 = chunk['price'].quantile(0.01)
        q99 = chunk['price'].quantile(0.99)
        chunk = chunk[(chunk['price'] >= q01) & (chunk['price'] <= q99)]
        stats['outliers_removed'] += before_price - len(chunk)
        
        # 7. Create features
        chunk['date'] = chunk['event_time'].dt.date.astype(str)
        chunk['hour'] = chunk['event_time'].dt.hour.astype('int16')
        chunk['day_of_week'] = chunk['event_time'].dt.dayofweek.astype('int8')
        chunk['day_name'] = chunk['event_time'].dt.day_name()
        chunk['month'] = chunk['event_time'].dt.month.astype('int8')
        chunk['year'] = chunk['event_time'].dt.year.astype('int16')
        
        # Category hierarchy
        cat_parts = chunk['category_code'].str.split('.', expand=True)
        if len(cat_parts.columns) > 0:
            chunk['category_level1'] = cat_parts[0].fillna('unknown')
        else:
            chunk['category_level1'] = 'unknown'
        if len(cat_parts.columns) > 1:
            chunk['category_level2'] = cat_parts[1].fillna('unknown')
        else:
            chunk['category_level2'] = 'unknown'
        if len(cat_parts.columns) > 2:
            chunk['category_level3'] = cat_parts[2].fillna('unknown')
        else:
            chunk['category_level3'] = 'unknown'
        
        chunk['source_file'] = csv_file
        
        # 8. Load to SQLite
        chunk['event_time'] = chunk['event_time'].astype(str)
        chunk.to_sql('events', conn, if_exists='append', index=False)
        
        total_rows_file += len(chunk)
        stats['total_rows'] += len(chunk)
        
        if (chunk_num + 1) % 5 == 0:
            print(f'  Chunk {chunk_num + 1}: {total_rows_file:,} rows processed')
        
        del chunk
        gc.collect()
    
    print(f'  -> {csv_file}: {total_rows_file:,} rows')

conn.commit()
conn.close()

print('\n' + '='*50)
print('EXTRACT + TRANSFORM + LOAD COMPLETE')
print('='*50)


Processing: 2019-Oct.csv


  Chunk 5: 465,178 rows processed


  Chunk 10: 929,110 rows processed


  Chunk 15: 1,392,649 rows processed


  Chunk 20: 1,860,084 rows processed


...

  Chunk 25: 2,319,236 rows processed


  Chunk 30: 2,782,684 rows processed


  Chunk 35: 3,247,378 rows processed


  Chunk 40: 3,712,048 rows processed


  -> 2020-Feb.csv: 3,730,508 rows

EXTRACT + TRANSFORM + LOAD COMPLETE


In [3]:
# Final statistics
conn = get_db_connection()
final_count = pd.read_sql_query('SELECT COUNT(*) as c FROM events', conn)['c'][0]
conn.close()

print('\n📊 FINAL STATISTICS:')
print(f'   Total rows in database: {final_count:,}')
print(f'   Duplicates removed: {stats["duplicates_removed"]:,}')
print(f'   Outliers removed: {stats["outliers_removed"]:,}')
print(f'   Sessions filtered: {stats["sessions_validated"]:,}')
print(f'\n📁 Database: {Config.DB_PATH}')
print(f'   Size: {Config.DB_PATH.stat().st_size / (1024*1024):.2f} MB')


📊 FINAL STATISTICS:
   Total rows in database: 38,254,680
   Duplicates removed: 1,109,082
   Outliers removed: 324,073
   Sessions filtered: 4,085

📁 Database: data\bigdata.db
   Size: 7047.54 MB


In [4]:
# Verify data
conn = get_db_connection()
sample = pd.read_sql_query('SELECT * FROM events LIMIT 5', conn)
print('Sample data:')
display(sample)
conn.close()

Sample data:


,event_id,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,source_file,date,hour,day_of_week,day_name,month,year,category_level1,category_level2,category_level3
0,1,2019-10-01 00:00:00+00:00,cart,5773203,1.487580e+18,unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885,2019-Oct.csv,2019-10-01,0,1,Tuesday,10,2019,unknown,unknown,unknown
1,2,2019-10-01 00:00:03+00:00,cart,5773353,1.487580e+18,unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885,2019-Oct.csv,2019-10-01,0,1,Tuesday,10,2019,unknown,unknown,unknown
2,3,2019-10-01 00:00:07+00:00,cart,5881589,2.151191e+18,unknown,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9,2019-Oct.csv,2019-10-01,0,1,Tuesday,10,2019,unknown,unknown,unknown
3,4,2019-10-01 00:00:07+00:00,cart,5723490,1.487580e+18,unknown,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885,2019-Oct.csv,2019-10-01,0,1,Tuesday,10,2019,unknown,unknown,unknown
4,5,2019-10-01 00:00:15+00:00,cart,5881449,1.487580e+18,unknown,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9,2019-Oct.csv,2019-10-01,0,1,Tuesday,10,2019,unknown,unknown,unknown


## 3. Summary

In [5]:
print('ETL Pipeline completed!')
print(f'\nOutput:')
print(f'  - SQLite DB: data/bigdata.db')
print(f'  - Table: events')
print(f'  - Rows: {final_count:,}')

ETL Pipeline completed!

Output:
  - SQLite DB: data/bigdata.db
  - Table: events
  - Rows: 38,254,680
